# MOD-05 · NB-06 — ICD Coding & Medication Extraction
### Health Analytics with Python · Module 05: NLP for Clinical Text
---
**Learning objectives**
- Build a TF-IDF ICD-10 code suggester from clinical text
- Extract structured medication information (drug, dose, route, frequency)
- Implement admission-to-discharge medication reconciliation
- Build a drug-drug interaction checker
- Validate extracted codes against ICD-10 format rules

**Estimated time:** 2.5 hours | **Level:** Advanced | **Libraries:** `re`, `pandas`, `sklearn`

## 1. Setup & ICD-10 Reference Table

In [ ]:
import re, os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
warnings.filterwarnings("ignore")
os.makedirs("/tmp/mod05", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})

ICD10_REF = pd.DataFrame([
    {"code":"I50.20","desc":"Unspecified systolic heart failure",
     "kw":"systolic heart failure reduced ejection fraction HFrEF EF low BNP furosemide"},
    {"code":"I50.30","desc":"Unspecified diastolic heart failure",
     "kw":"diastolic heart failure preserved ejection HFpEF"},
    {"code":"I50.9", "desc":"Unspecified heart failure",
     "kw":"heart failure congestive CHF decompensated BNP edema"},
    {"code":"I21.19","desc":"STEMI other coronary artery",
     "kw":"STEMI myocardial infarction ST elevation troponin cath lab PCI stent"},
    {"code":"I21.4", "desc":"Non-ST elevation MI",
     "kw":"NSTEMI troponin elevated demand ischemia non-ST elevation ACS"},
    {"code":"I48.91","desc":"Unspecified atrial fibrillation",
     "kw":"atrial fibrillation AFib arrhythmia rate control anticoagulation"},
    {"code":"J44.1", "desc":"COPD with acute exacerbation",
     "kw":"COPD exacerbation bronchospasm wheezes steroids albuterol ipratropium"},
    {"code":"J18.9", "desc":"Pneumonia unspecified",
     "kw":"pneumonia consolidation fever WBC antibiotics community acquired CAP"},
    {"code":"J26.99","desc":"Other pulmonary embolism",
     "kw":"pulmonary embolism PE DVT anticoagulation rivaroxaban heparin RV strain"},
    {"code":"E11.10","desc":"T2DM with DKA without coma",
     "kw":"diabetic ketoacidosis DKA anion gap metabolic acidosis insulin drip glucose"},
    {"code":"E11.65","desc":"T2DM with hyperglycemia",
     "kw":"uncontrolled diabetes hyperglycemia HbA1c elevated glucose A1c"},
    {"code":"N17.9", "desc":"Acute kidney failure",
     "kw":"acute kidney injury AKI creatinine elevated BUN nephrotoxins oliguria"},
    {"code":"N18.6", "desc":"End-stage renal disease",
     "kw":"ESRD hemodialysis dialysis end stage renal disease potassium"},
    {"code":"A41.9", "desc":"Sepsis unspecified organism",
     "kw":"sepsis septic shock bacteremia lactate broad spectrum antibiotics norepinephrine"},
    {"code":"I10",   "desc":"Essential hypertension",
     "kw":"hypertension HTN blood pressure elevated antihypertensive"},
])
print(f"ICD-10 reference: {len(ICD10_REF)} codes")
display(ICD10_REF[["code","desc"]].head(8))


## 2. TF-IDF ICD-10 Code Suggester

In [ ]:
tfidf_icd = TfidfVectorizer(ngram_range=(1,2), max_features=500)
icd_matrix = tfidf_icd.fit_transform(ICD10_REF["kw"] + " " + ICD10_REF["desc"])

def suggest_icd(text, top_k=5, threshold=0.12):
    vec    = tfidf_icd.transform([text.lower()])
    scores = cosine_similarity(vec, icd_matrix).flatten()
    top_idx = scores.argsort()[-top_k:][::-1]
    return [{"code":ICD10_REF.iloc[i]["code"],
             "desc":ICD10_REF.iloc[i]["desc"],
             "score":round(float(scores[i]),3)}
            for i in top_idx if scores[i] >= threshold]

test_notes = [
    "Acute decompensated heart failure, BNP 1450, EF 30%, IV furosemide started.",
    "STEMI inferior wall, troponin 2.4, EKG ST elevation, PCI with stent placement.",
    "COPD exacerbation, diffuse wheezes, albuterol nebs, methylprednisolone IV.",
    "Diabetic ketoacidosis, glucose 520, anion gap 24, insulin drip initiated.",
    "Septic shock, lactate 4.2, norepinephrine, vancomycin and meropenem started.",
]
print("ICD-10 Code Suggestions:\n")
for note in test_notes:
    suggestions = suggest_icd(note, top_k=3)
    print(f"  Note: '{note[:60]}'")
    for s in suggestions:
        print(f"    {s['code']:8s} ({s['score']:.3f}) — {s['desc']}")
    print()


## 3. Structured Medication Extraction

In [ ]:
DRUG_LIST = sorted([
    "metoprolol","carvedilol","bisoprolol","atenolol","labetalol",
    "lisinopril","enalapril","ramipril","losartan","valsartan","sacubitril",
    "amlodipine","nifedipine","diltiazem","verapamil",
    "furosemide","torsemide","spironolactone","hydrochlorothiazide",
    "warfarin","rivaroxaban","apixaban","dabigatran","edoxaban",
    "aspirin","clopidogrel","ticagrelor","prasugrel",
    "heparin","enoxaparin","fondaparinux",
    "digoxin","amiodarone","sotalol",
    "atorvastatin","rosuvastatin","simvastatin","pravastatin",
    "nitroglycerin","nitroprusside","norepinephrine","vasopressin",
    "vancomycin","linezolid","ceftriaxone","cefazolin","cefepime",
    "piperacillin-tazobactam","meropenem","imipenem","ertapenem",
    "azithromycin","clarithromycin","doxycycline","ciprofloxacin","levofloxacin",
    "trimethoprim-sulfamethoxazole","nitrofurantoin",
    "metformin","glipizide","glimepiride","empagliflozin","dapagliflozin",
    "sitagliptin","liraglutide","semaglutide",
    "insulin","insulin glargine","insulin detemir","insulin aspart","insulin lispro",
    "albuterol","ipratropium","tiotropium","fluticasone","salmeterol","budesonide",
    "prednisone","methylprednisolone","dexamethasone","hydrocortisone",
    "omeprazole","pantoprazole","lansoprazole","famotidine",
    "ondansetron","metoclopramide","acetaminophen","ibuprofen","ketorolac",
    "morphine","hydromorphone","fentanyl","oxycodone","tramadol",
    "midazolam","lorazepam","propofol","dexmedetomidine",
    "fluconazole","micafungin","acyclovir","levothyroxine",
], key=len, reverse=True)

DOSE_UNIT = r'(?:mg|mcg|g|mL|units?|IU|mEq)'
ROUTE_PAT = r'(?:PO|IV|IM|SQ|SC|SL|inhaled|topical|PR)'
FREQ_PAT  = r'(?:QD|BID|TID|QID|QAM|QHS|q\d+h|PRN|once daily|twice daily|daily|weekly)'
DRUG_PAT  = r'\b(' + '|'.join(re.escape(d) for d in DRUG_LIST) + r')\b'
MED_PAT   = (DRUG_PAT
             + r'(?:[\s,]+(\d+(?:\.\d+)?)\s*(' + DOSE_UNIT + r'))?'
             + r'(?:[\s,]+(' + ROUTE_PAT + r'))?'
             + r'(?:[\s,]+(' + FREQ_PAT  + r'))?')

def extract_meds(text):
    seen = set(); meds = []
    for m in re.finditer(MED_PAT, text, re.IGNORECASE):
        drug = m.group(1).lower()
        if drug not in seen:
            seen.add(drug)
            meds.append({"drug":drug,"dose":(m.group(2) or "").strip(),
                         "unit":(m.group(3) or "").strip(),
                         "route":(m.group(4) or "").upper(),
                         "freq": (m.group(5) or "").upper()})
    return meds

test_med_notes = [
    "Patient is on metoprolol 50 mg PO BID, lisinopril 10 mg PO QD, furosemide 40 mg PO QAM, warfarin 5 mg PO QD.",
    "Started vancomycin 1250 mg IV q12h, piperacillin-tazobactam 4.5 g IV q8h, fluconazole 400 mg IV QD.",
    "Insulin glargine 20 units SQ QHS, metformin 1000 mg PO BID, empagliflozin 10 mg PO QD.",
]
print("Structured medication extraction:\n")
for note in test_med_notes:
    meds = extract_meds(note)
    print(f"Note: '{note[:65]}'")
    for m in meds:
        print(f"  {m['drug']:30s} | {m['dose']:6s} {m['unit']:6s} | {m['route']:4s} | {m['freq']}")
    print()


## 4. Medication Reconciliation

In [ ]:
def reconcile_medications(admit_text, dc_text):
    admit_meds = {m["drug"]:m for m in extract_meds(admit_text)}
    dc_meds    = {m["drug"]:m for m in extract_meds(dc_text)}
    admit_set  = set(admit_meds.keys())
    dc_set     = set(dc_meds.keys())
    continued    = sorted(admit_set & dc_set)
    discontinued = sorted(admit_set - dc_set)
    new_at_dc    = sorted(dc_set - admit_set)
    dose_changed = sorted(d for d in continued
                          if admit_meds[d]["dose"] != dc_meds[d]["dose"]
                          and admit_meds[d]["dose"] and dc_meds[d]["dose"])
    return {"continued":continued,"discontinued":discontinued,
            "new_at_dc":new_at_dc,"dose_changed":dose_changed,
            "admit_meds":admit_meds,"dc_meds":dc_meds}

ADMIT_MEDS = (
    "metoprolol 50 mg PO BID, lisinopril 10 mg PO QD, furosemide 40 mg PO QAM, "
    "warfarin 5 mg PO QD, atorvastatin 40 mg PO QHS, metformin 1000 mg PO BID, aspirin 81 mg PO QD."
)
DC_MEDS = (
    "metoprolol 25 mg PO BID, lisinopril 10 mg PO QD, furosemide 80 mg PO QAM, "
    "rivaroxaban 20 mg PO QD, atorvastatin 40 mg PO QHS, metformin 1000 mg PO BID, "
    "aspirin 81 mg PO QD, spironolactone 25 mg PO QD, sacubitril 24 mg PO BID."
)

recon = reconcile_medications(ADMIT_MEDS, DC_MEDS)
print("MEDICATION RECONCILIATION REPORT")
print("="*55)
print(f"Admission: {len(recon['admit_meds'])} meds | Discharge: {len(recon['dc_meds'])} meds")
print(f"\nContinued ({len(recon['continued'])}):")
for m in recon["continued"]: print(f"  checkmark {m}")
print(f"\nDiscontinued ({len(recon['discontinued'])}):")
for m in recon["discontinued"]: print(f"  removed   {m}")
print(f"\nNew at discharge ({len(recon['new_at_dc'])}):")
for m in recon["new_at_dc"]: print(f"  added     {m}")
print(f"\nDose changed ({len(recon['dose_changed'])}):")
for m in recon["dose_changed"]:
    a = recon["admit_meds"][m]; d = recon["dc_meds"][m]
    print(f"  changed   {m}: {a['dose']} {a['unit']} -> {d['dose']} {d['unit']}")


## 5. Drug-Drug Interaction Checker

In [ ]:
DDI_REF = [
    ("warfarin",    "aspirin",          "HIGH", "Increased bleeding risk — monitor INR closely"),
    ("warfarin",    "ciprofloxacin",    "HIGH", "CYP2C9 inhibition — warfarin effect increased"),
    ("warfarin",    "fluconazole",      "HIGH", "Azole antifungals inhibit warfarin metabolism"),
    ("digoxin",     "amiodarone",       "HIGH", "Digoxin toxicity — reduce dose by 50%"),
    ("heparin",     "enoxaparin",       "HIGH", "Concurrent anticoagulants — serious bleeding risk"),
    ("metformin",   "contrast",         "MOD",  "Hold 48h post-contrast — nephropathy risk"),
    ("lisinopril",  "spironolactone",   "MOD",  "Hyperkalaemia risk — monitor K+ closely"),
    ("metoprolol",  "diltiazem",        "MOD",  "Additive bradycardia and hypotension"),
    ("aspirin",     "ibuprofen",        "MOD",  "NSAIDs reduce aspirin cardioprotection"),
]

def check_ddi(medication_list):
    med_names = {m["drug"].lower() for m in medication_list}
    flagged = []
    for d1,d2,severity,msg in DDI_REF:
        if d1 in med_names and d2 in med_names:
            flagged.append({"drug1":d1,"drug2":d2,"severity":severity,"message":msg})
    return flagged

complex_note = (
    "warfarin 5 mg QD, aspirin 81 mg QD, metformin 500 mg BID, "
    "lisinopril 10 mg QD, spironolactone 25 mg QD, digoxin 0.125 mg QD, "
    "amiodarone 200 mg QD, fluconazole 400 mg IV QD."
)
meds_list = extract_meds(complex_note)
interactions = check_ddi(meds_list)
print(f"Medications: {[m['drug'] for m in meds_list]}")
print(f"\nDrug-Drug Interactions ({len(interactions)} flagged):\n")
for i in interactions:
    icon = "HIGH_RISK" if i["severity"]=="HIGH" else "MOD_RISK"
    print(f"  [{icon}] {i['drug1']} + {i['drug2']}")
    print(f"           {i['message']}")


## 6. ICD-10 Code Validation

In [ ]:
def validate_icd(code):
    code = str(code).strip().upper()
    valid = bool(re.match(r'^[A-Z]\d{2}(\.\d{0,4})?$', code))
    match = ICD10_REF[ICD10_REF["code"]==code]
    return {"code":code,"valid_format":valid,"in_reference":len(match)>0,
            "desc":match.iloc[0]["desc"] if len(match)>0 else "Not in reference"}

test_codes = ["I50.9","E11.10","A41.9","INVALID","I10","XYZ1","J44.1","N17.9","I50.20"]
print("ICD-10 Code Validation:\n")
print(f"{'Code':10s} {'Format':10s} {'In Ref':8s} {'Description'}")
print("-"*65)
for code in test_codes:
    r = validate_icd(code)
    fmt = "Valid" if r["valid_format"] else "INVALID"
    ref = "Yes"  if r["in_reference"] else "No"
    print(f"  {code:8s} {fmt:10s} {ref:8s} {r['desc'][:45]}")


## 7. Medication Completeness Audit

In [ ]:
def completeness_audit(text):
    meds = extract_meds(text)
    complete   = [m for m in meds if m["freq"] or m["route"]]
    incomplete = [m for m in meds if not m["freq"] and not m["route"]]
    print(f"Total medications : {len(meds)}")
    print(f"Complete records  : {len(complete)}")
    print(f"Incomplete records: {len(incomplete)}")
    if incomplete:
        print("\nMedications missing route AND frequency:")
        for m in incomplete:
            print(f"  {m['drug']:25s} dose={m['dose']} {m['unit']} -- INCOMPLETE")
    return incomplete

sample = (
    "Patient is on atorvastatin 40 mg, metformin 500 mg PO BID, "
    "aspirin 81 mg, lisinopril 10 mg PO QD, furosemide."
)
completeness_audit(sample)


## 8. Knowledge Check
1. ICD-10 suggester gives I50.9 for EF=30%, systolic dysfunction. Is I50.20 more specific? Why does specificity matter for DRG billing?
2. Your medication extractor misses 'Lasix 40 mg' (brand name). How do you fix this?
3. Reconciliation found warfarin discontinued, rivaroxaban added. What clinical monitoring is needed at the transition?
4. DDI checker flags warfarin + aspirin HIGH — but this combination is sometimes intentional. How do you handle this in a real system?
5. Add route-specific dose validation: oral doses should not exceed typical maximum; flag outliers.

In [ ]:
# Exercise 5 — dose range validation
DOSE_LIMITS_ORAL = {
    "metformin": (250, 2550), "furosemide": (20, 600),
    "lisinopril": (2.5, 40),  "warfarin": (1, 15),
    "aspirin": (81, 325),     "atorvastatin": (10, 80),
}

def validate_doses(text):
    meds = extract_meds(text)
    flags = []
    for m in meds:
        drug = m["drug"]
        if drug in DOSE_LIMITS_ORAL and m["dose"] and m["route"] in ["PO",""]:
            try:
                dose = float(m["dose"])
                lo, hi = DOSE_LIMITS_ORAL[drug]
                if not (lo <= dose <= hi):
                    flags.append({"drug":drug,"dose":dose,"unit":m["unit"],
                                  "expected":f"{lo}-{hi} mg",
                                  "flag":"ABOVE MAX" if dose>hi else "BELOW MIN"})
            except ValueError:
                pass
    return flags

test_doses = (
    "metformin 3000 mg PO BID, furosemide 20 mg PO QD, "
    "aspirin 81 mg PO QD, lisinopril 80 mg PO QD, warfarin 5 mg PO QD."
)
flags = validate_doses(test_doses)
print(f"Dose validation flags ({len(flags)} found):")
for f in flags:
    print(f"  WARNING: {f['drug']:15s} {f['dose']} {f['unit']:5s} -- {f['flag']} (expected {f['expected']})")


---
**Next → NB-07: Clinical Text Summarisation & Zero-Shot NLP**